In [1]:
import polars as pl
import pathlib
import process_raw_icem

### Reading Raw Census Data

Converts raw census csv files to parquet, with some additional processing to clean the files, reduce size for memory efficiency etc

In [2]:
# set list of columns to read from raw census data and output as processed parquet files - keep list broad to ensure all columns for 
# future analysis are available but limited enough for files to fit in memory
col_list = [
    "AdminCnty", "Age", "BP_EW", "BP_NS", "BpCmty", "BpCNTI", "BpCNTRY", "BpCnty", "BpCtry",
    "BpID", "Bpstring", "BTCode", "BuildType", "Cage", "Cen_Num", "Censusref", "CenYear",
    "CFU", "CFUsize", "Cond", "ConParID", "Country", "disab", "Distance", "DocType",
    "EM_Offsp", "Employ", "EnuDist", "Father", "H_Age", "H_CFU", "H_Mar", "H_Occ",
    "H_Sex", "HHD", "hid", "Household", "imageREF", "Inmates", "InstDesc", "InstName",
    "Intage", "LatLong", "Mar", "Mother", "Nationality", "NM_Offsp", "Non_Rels",
    "NoOfRooms", "Noofroomscode", "Occ", "Occode", "ParID", "Parish", "pid", "RD", "recid",
    "RegCnty", "RegDist", "Rela", "Relat", "Relats", "RSD", "Schedule", "Servts", "Sex",
    "Spouse", "StanPlace", "SubDist", "TotalPersons", "Visitors"
]

### Set paths to input and output data

Code assumes that census files are saved as year.txt, e.g. 1911.txt.

In [ ]:
raw_census_dir = pathlib.Path("path/to/censusdata")
processed_census_dir = pathlib.Path("path/to/censusdata/processed")


processed_census_dir.mkdir(exist_ok=True)

In [ ]:
# Set census years to iterate over
census_years = ["1851", "1861", "1871", "1881", "1891", "1901", "1911"]

### Read and process raw census csv files by census year

- Scan each raw census csv and output as parquet (massively reduces size of census file on disk)
- Process each parquet file making corrections, downcasting dtypes to improve memory performance
- Write processed data to parquet files partitioned on census country.

In [5]:
for year in census_years:
    
    print(year)
    
    raw_census_file = raw_census_dir.joinpath(f"{year}.txt")
    raw_parquet_file = raw_census_dir.joinpath(f"{year}.parquet")
    processed_parquet_dir = processed_census_dir.joinpath(year)


    pl.scan_csv(raw_census_file, separator="\t", encoding="utf8-lossy", quote_char=None, infer_schema_length=100000,null_values=[" ", " - ", " . "]
            ).sink_parquet(raw_parquet_file)
    
    df = pl.read_parquet(raw_parquet_file, columns=col_list,)

    df = process_raw_icem.fix_encoding_errors(df, )
    df = df.with_columns(pl.col(pl.String).str.strip_chars())

    df = process_raw_icem.replace_numeric_null_values(df, cols_to_ignore = ["recid", "hid"], null_value = 999999)

    df = process_raw_icem.downcast_integers(df, )
    df = process_raw_icem.downcast_floats(df, )
    df = process_raw_icem.create_categoricals(df, )

    df = process_raw_icem.fix_recid_errors(df, "hid_lkp_recid_fix.json", year)

    df.write_parquet(processed_parquet_dir, partition_by="Country")

1851
1861
1871
1881
1891
1901
1911
